# Icy Tower — trening PPO na Google Colab

## Przed startem
1. **Runtime → Change runtime type → T4 GPU** (zalecane)
2. Wybierz **jedną** metodę pobrania kodu w komórce 2: `git` albo `upload`
3. Uruchamiaj komórki **po kolei** (Shift+Enter)

Repo: https://github.com/tomrat04/icy_tower

In [ ]:
# ========== KONFIGURACJA ==========
SOURCE = "git"  # "git" lub "upload"
REPO_URL = "https://github.com/tomrat04/icy_tower.git"
BRANCH = "master"

DRIVE_DIR = "/content/drive/MyDrive/icy_tower"
REPO_DIR = "/content/icy_tower"

N_ENVS = 24
TIMESTEPS = 2_000_000
RUN_NAME = "colab_ppo"
N_EVAL = 30

In [ ]:
# Zależności (cudzysłowy przy [extra] są wymagane!)
!pip install -q pygame gymnasium "stable-baselines3[extra]" tensorboard tqdm rich

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

if SOURCE == "git":
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
elif SOURCE == "upload":
    from google.colab import files
    print("Wybierz plik ZIP z projektem (cały folder icy_tower).")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall("/content")
    # Szukaj train.py — obsługa różnych struktur zipa
    candidates = list(Path("/content").rglob("train.py"))
    train_paths = [p for p in candidates if p.parent.name == "icy_tower" or "icy_tower" in str(p)]
    if not train_paths:
        raise FileNotFoundError("W zipie brak train.py — spakuj cały folder projektu icy_tower")
    found = train_paths[0].parent
    if found != Path(REPO_DIR):
        if Path(REPO_DIR).exists():
            shutil.rmtree(REPO_DIR)
        shutil.move(str(found), REPO_DIR)
else:
    raise ValueError('SOURCE musi być "git" lub "upload"')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("Pliki:", sorted(os.listdir(REPO_DIR))[:12])

In [ ]:
# Test importów — jeśli tu błąd, trening też nie ruszy
import torch
from icy_tower.env import IcyTowerEnv
from icy_tower.observations import OBS_DIM

env = IcyTowerEnv(seed=0)
obs, info = env.reset()
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("OBS_DIM:", OBS_DIM, "obs shape:", obs.shape)
print("start_level:", info.get("start_level"))
env.close()
print("OK — środowisko działa")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Path(DRIVE_DIR, "models").mkdir(parents=True, exist_ok=True)
Path(DRIVE_DIR, "logs").mkdir(parents=True, exist_ok=True)
print("Zapis modeli:", f"{DRIVE_DIR}/models")
print("Logi TB:", f"{DRIVE_DIR}/logs")

## Trening od zera
Pomiń, jeśli kontynuujesz z komórki poniżej.

In [ ]:
import subprocess

cmd = [
    "python", "train.py",
    "--n-envs", str(N_ENVS),
    "--timesteps", str(TIMESTEPS),
    "--n-eval-episodes", str(N_EVAL),
    "--save-dir", f"{DRIVE_DIR}/models",
    "--tb-log", f"{DRIVE_DIR}/logs",
    "--run-name", RUN_NAME,
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

## Kontynuacja po zerwaniu sesji

In [ ]:
LOAD_MODEL = f"{DRIVE_DIR}/models/icy_ppo_final"  # bez .zip
ADDITIONAL_STEPS = 1_000_000

cmd = [
    "python", "train.py",
    "--load-model", LOAD_MODEL,
    "--additional-timesteps", str(ADDITIONAL_STEPS),
    "--n-envs", str(N_ENVS),
    "--n-eval-episodes", str(N_EVAL),
    "--save-dir", f"{DRIVE_DIR}/models",
    "--tb-log", f"{DRIVE_DIR}/logs",
    "--run-name", f"{RUN_NAME}_resume",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/logs

## Pobierz model na PC
Lokalnie: `python test.py --model ścieżka/do/icy_ppo_final`

In [ ]:
from google.colab import files
from pathlib import Path

model_zip = Path(DRIVE_DIR) / "models" / "icy_ppo_final.zip"
if not model_zip.exists():
    raise FileNotFoundError(f"Brak {model_zip} — najpierw uruchom trening")
files.download(str(model_zip))